# 0. Setup

Run this cell first.  
If you are using Colab, select:

`Runtime` → `Change runtime type` → `T4 GPU`

A GPU is recommended for ESM embedding extraction, but the model is intentionally small enough to run on Colab.

In [ ]:
# Install dependencies.
# This may take a few minutes in a fresh Colab runtime.

# !pip install pytdc rdkit fair-esm
# !pip -q install "transformers>=4.40,<5" huggingface-hub safetensors

In [ ]:
import os
import random
import pickle
import copy
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from scipy.stats import pearsonr, spearmanr

from tdc.multi_pred import DTI

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

import esm

In [ ]:
# Reproducibility

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# 1. Load KIBA from TDC

TDC KIBA is a DTA/DTI regression dataset.

Each row contains:

- `Drug`: compound SMILES string
- `Target`: protein amino acid sequence
- `Y`: binding affinity / activity score

In [ ]:
DATASET_NAME = "KIBA"

data = DTI(name=DATASET_NAME, path="./data/KIBA")
df = data.get_data()

print("Dataset:", DATASET_NAME)
print(df.shape)
display(df.head())
print(df.columns.tolist())

In [ ]:
# Basic sanity checks

required_cols = ["Drug", "Target", "Y"]
for col in required_cols:
    assert col in df.columns, f"Missing required column: {col}"

print("Number of DTA pairs:", len(df))
print("Number of unique drugs:", df["Drug"].nunique())
print("Number of unique targets:", df["Target"].nunique())
print()
print("Y summary:")
display(df["Y"].describe())

# 2. Create random and cold-start splits

We will use three split settings.

| Split | Meaning |
|---|---|
| Random | Drug–target pairs are randomly split. Drugs and proteins may appear in both train and test. |
| Cold drug | Test drugs do not appear in train. |
| Cold target | Test proteins do not appear in train. |

The cold-start splits are more realistic and more challenging.

In [ ]:
FRAC = [0.7, 0.1, 0.2]

splits = {
    "random": data.get_split(method="random", seed=SEED, frac=FRAC),
    "cold_drug": data.get_split(method="cold_split", column_name="Drug", seed=SEED, frac=FRAC),
    "cold_target": data.get_split(method="cold_split", column_name="Target", seed=SEED, frac=FRAC),
}

for split_name, split in splits.items():
    print(f"\n[{split_name}]")
    for part in ["train", "valid", "test"]:
        print(part, split[part].shape)

In [ ]:
def check_overlap(split, column):
    train_set = set(split["train"][column])
    valid_set = set(split["valid"][column])
    test_set = set(split["test"][column])
    return {
        "train_valid_overlap": len(train_set & valid_set),
        "train_test_overlap": len(train_set & test_set),
        "valid_test_overlap": len(valid_set & test_set),
    }

print("Cold drug overlap check on Drug:")
print(check_overlap(splits["cold_drug"], "Drug"))

print("\nCold target overlap check on Target:")
print(check_overlap(splits["cold_target"], "Target"))

# 3. Feature extraction

We will use:

## Drug feature

Morgan fingerprint:

- Radius: 2
- Number of bits: 1024

## Protein feature

ESM-2 small model:

- Model: `esm2_t6_8M_UR50D`
- Embedding dimension: 320
- Pooling: mean pooling over residue representations

For long proteins, we truncate to the first 1022 amino acids.  
This is a baseline simplification and should be discussed as a limitation.

In [ ]:
CACHE_DIR = Path("./data/cache")
CACHE_DIR.mkdir(exist_ok=True)

DRUG_FP_BITS = 1024
DRUG_FP_RADIUS = 2

PROTEIN_MAX_LEN = 1022
ESM_REPR_LAYER = 6
ESM_BATCH_SIZE = 8

In [ ]:
def smiles_to_morgan_fp(smiles, radius=2, n_bits=1024):
    """Convert a SMILES string to a Morgan fingerprint numpy vector."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits, dtype=np.float32)

    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr.astype(np.float32)


def build_or_load_drug_features(df, cache_path):
    if Path(cache_path).exists():
        print(f"Loading cached drug features from {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    print("Building Morgan fingerprints...")
    drug_features = {}
    unique_smiles = sorted(df["Drug"].unique())

    for smiles in tqdm(unique_smiles):
        drug_features[smiles] = smiles_to_morgan_fp(
            smiles,
            radius=DRUG_FP_RADIUS,
            n_bits=DRUG_FP_BITS,
        )

    with open(cache_path, "wb") as f:
        pickle.dump(drug_features, f)

    return drug_features


drug_features = build_or_load_drug_features(
    df,
    CACHE_DIR / f"{DATASET_NAME.lower()}_drug_morgan_r{DRUG_FP_RADIUS}_{DRUG_FP_BITS}.pkl",
)

example_drug_vec = next(iter(drug_features.values()))
print("Drug feature dim:", example_drug_vec.shape)

In [ ]:
def build_or_load_protein_esm_embeddings(df, cache_path, batch_size=8):
    """Build or load ESM-2 mean-pooled protein embeddings.

    Keys are original full protein sequences.
    Values are float32 numpy arrays of shape [320].
    """
    if Path(cache_path).exists():
        print(f"Loading cached protein ESM embeddings from {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    print("Loading ESM-2 model...")
    esm_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    batch_converter = alphabet.get_batch_converter()

    esm_model.eval()
    esm_model = esm_model.to(device)

    unique_targets = sorted(df["Target"].unique())
    protein_embeddings = {}

    print(f"Embedding {len(unique_targets)} unique protein sequences...")

    for start in tqdm(range(0, len(unique_targets), batch_size)):
        batch_original = unique_targets[start:start + batch_size]

        # Truncate sequences for baseline.
        # We keep original sequences as dictionary keys.
        batch_data = [
            (f"protein_{start + i}", seq[:PROTEIN_MAX_LEN])
            for i, seq in enumerate(batch_original)
        ]

        batch_labels, batch_strs, batch_tokens = batch_converter(batch_data)
        batch_tokens = batch_tokens.to(device)

        with torch.no_grad():
            results = esm_model(
                batch_tokens,
                repr_layers=[ESM_REPR_LAYER],
                return_contacts=False,
            )

        token_representations = results["representations"][ESM_REPR_LAYER].detach().cpu()

        for i, original_seq in enumerate(batch_original):
            trunc_len = min(len(original_seq), PROTEIN_MAX_LEN)

            # token_representations includes special BOS/EOS tokens.
            # Residue tokens are positions 1 to trunc_len inclusive.
            residue_repr = token_representations[i, 1:trunc_len + 1]
            mean_repr = residue_repr.mean(dim=0).numpy().astype(np.float32)

            protein_embeddings[original_seq] = mean_repr

    with open(cache_path, "wb") as f:
        pickle.dump(protein_embeddings, f)

    return protein_embeddings


protein_features = build_or_load_protein_esm_embeddings(
    df,
    CACHE_DIR / f"{DATASET_NAME.lower()}_protein_esm2_t6_mean_len{PROTEIN_MAX_LEN}.pkl",
    batch_size=ESM_BATCH_SIZE,
)

example_protein_vec = next(iter(protein_features.values()))
print("Protein feature dim:", example_protein_vec.shape)

# 4. Dataset and feature matrix construction

For each DTA pair:

\[
x = [\text{Morgan fingerprint}; \text{ESM protein embedding}]
\]

\[
y = \text{affinity score}
\]

In [ ]:
def build_xy(dataframe, drug_features, protein_features):
    drug_x = np.stack([drug_features[smi] for smi in dataframe["Drug"].values]).astype(np.float32)
    protein_x = np.stack([protein_features[seq] for seq in dataframe["Target"].values]).astype(np.float32)
    y = dataframe["Y"].values.astype(np.float32)
    return drug_x, protein_x, y


class DTADataset(Dataset):
    def __init__(self, drug_x, protein_x, y):
        self.drug_x = torch.tensor(drug_x, dtype=torch.float32)
        self.protein_x = torch.tensor(protein_x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.drug_x[idx], self.protein_x[idx], self.y[idx]

In [ ]:
# Check feature dimensions with a small example

drug_x_tmp, protein_x_tmp, y_tmp = build_xy(splits["random"]["train"].head(5), drug_features, protein_features)
print("Example drug X shape:", drug_x_tmp.shape)
print("Example protein X shape:", protein_x_tmp.shape)
print("Example y shape:", y_tmp.shape)

DRUG_DIM = drug_x_tmp.shape[1]
PROTEIN_DIM = protein_x_tmp.shape[1]
print("Drug dimension:", DRUG_DIM)
print("Protein dimension:", PROTEIN_DIM)

# 5. Evaluation metrics

We report:

- RMSE
- MAE
- Pearson correlation
- Spearman correlation

Primary metric: **RMSE**  
Secondary metrics: **MAE, Pearson, Spearman**

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))

    # Correlations can fail if predictions are constant.
    try:
        pearson = pearsonr(y_true, y_pred).statistic
    except Exception:
        pearson = np.nan

    try:
        spearman = spearmanr(y_true, y_pred).statistic
    except Exception:
        spearman = np.nan

    return {
        "RMSE": float(rmse),
        "MAE": float(mae),
        "Pearson": float(pearson),
        "Spearman": float(spearman),
    }

# 6. Baseline model

Baseline:

```text
Morgan fingerprint 1024-d
+
ESM protein embedding 320-d
→ concat 1344-d
→ MLP
→ affinity prediction
```

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self, drug_dim, protein_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        input_dim = drug_dim + protein_dim

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 1),
        )

    def forward(self, drug_x, protein_x):
        x = torch.cat([drug_x, protein_x], dim=1)
        return self.net(x).squeeze(-1)

# 7. Training utilities

Important detail:

The label `Y` is standardized using **train split statistics only**.

During evaluation, predictions are transformed back to the original label scale before computing metrics.

In [ ]:
def predict(model, loader, device):
    model.eval()
    preds = []
    ys = []

    with torch.no_grad():
        for drug_x, protein_x, y in loader:
            drug_x = drug_x.to(device)
            protein_x = protein_x.to(device)
            pred = model(drug_x, protein_x).detach().cpu().numpy()
            preds.append(pred)
            ys.append(y.numpy())

    preds = np.concatenate(preds)
    ys = np.concatenate(ys)
    return ys, preds


def train_one_split(
    split_name,
    split,
    model_class,
    xy_builder=None,
    model_kwargs=None,
    epochs=50,
    batch_size=256,
    lr=1e-4,
    weight_decay=1e-4,
    patience=10,
):
    if model_kwargs is None:
        model_kwargs = {}

    print(f"\n===== Training on split: {split_name} =====")

    if xy_builder is None:
        def xy_builder(dataframe):
            return build_xy(dataframe, drug_features, protein_features)

    drug_x_train, protein_x_train, y_train_raw = xy_builder(split["train"])
    drug_x_valid, protein_x_valid, y_valid_raw = xy_builder(split["valid"])
    drug_x_test, protein_x_test, y_test_raw = xy_builder(split["test"])

    # Standardize labels using train statistics only.
    y_mean = y_train_raw.mean()
    y_std = y_train_raw.std() + 1e-8

    y_train = (y_train_raw - y_mean) / y_std
    y_valid = (y_valid_raw - y_mean) / y_std
    y_test = (y_test_raw - y_mean) / y_std

    train_loader = DataLoader(
        DTADataset(drug_x_train, protein_x_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    valid_loader = DataLoader(
        DTADataset(drug_x_valid, protein_x_valid, y_valid),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        DTADataset(drug_x_test, protein_x_test, y_test),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    drug_dim = drug_x_train.shape[1]
    protein_dim = protein_x_train.shape[1]
    model = model_class(drug_dim=drug_dim, protein_dim=protein_dim, **model_kwargs).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_valid_loss = float("inf")
    best_state = None
    wait = 0

    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for drug_x, protein_x, y in train_loader:
            drug_x = drug_x.to(device)
            protein_x = protein_x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            pred = model(drug_x, protein_x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        # Validation loss on standardized scale
        model.eval()
        valid_losses = []

        with torch.no_grad():
            for drug_x, protein_x, y in valid_loader:
                drug_x = drug_x.to(device)
                protein_x = protein_x.to(device)
                y = y.to(device)
                pred = model(drug_x, protein_x)
                loss = criterion(pred, y)
                valid_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        valid_loss = float(np.mean(valid_losses))

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
        })

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % 5 == 0:
            print(f"Epoch {epoch:03d} | train loss {train_loss:.4f} | valid loss {valid_loss:.4f}")

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)

    # Predict standardized values, then inverse transform to original scale.
    _, valid_pred_std = predict(model, valid_loader, device)
    _, test_pred_std = predict(model, test_loader, device)

    valid_pred = valid_pred_std * y_std + y_mean
    test_pred = test_pred_std * y_std + y_mean

    valid_metrics = compute_metrics(y_valid_raw, valid_pred)
    test_metrics = compute_metrics(y_test_raw, test_pred)

    print("Valid metrics:", valid_metrics)
    print("Test metrics:", test_metrics)

    result = {
        "Split": split_name,
        **test_metrics,
        "BestValidLoss_standardized_MSE": best_valid_loss,
        "EpochsTrained": len(history),
    }

    history_df = pd.DataFrame(history)

    return result, model, history_df

# 8. Run baseline experiments

This cell trains and evaluates the baseline model on:

1. Random split
2. Cold drug split
3. Cold target split

Expected pattern:

- Random split is usually easiest.
- Cold drug and cold target are usually harder.

In [ ]:
baseline_results = []
baseline_histories = {}

BASELINE_CONFIG = {
    "epochs": 50,
    "batch_size": 256,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "patience": 10,
    "model_kwargs": {
        "hidden_dim": 512,
        "dropout": 0.2,
    },
}

for split_name, split in splits.items():
    result, model, history_df = train_one_split(
        split_name=split_name,
        split=split,
        model_class=BaselineMLP,
        **BASELINE_CONFIG,
    )

    result["Model"] = "BaselineMLP"
    baseline_results.append(result)
    baseline_histories[split_name] = history_df

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df = baseline_results_df[
    ["Model", "Split", "RMSE", "MAE", "Pearson", "Spearman", "BestValidLoss_standardized_MSE", "EpochsTrained"]
]

display(baseline_results_df)

baseline_results_path = f"{DATASET_NAME.lower()}_baseline_results.csv"
baseline_results_df.to_csv(baseline_results_path, index=False)
print(f"Saved {baseline_results_path}")

# 9. Student task: improve the model

Your task is to improve the baseline model.

This section contains:

1. **A reference example cell**
   - This shows one possible improvement idea.
   - It is not used automatically for grading.
2. **Student feature / representation cell**
   - By default it reuses the baseline Morgan + ESM features.
   - You may replace them with other representations such as MolFormer embeddings, alternative protein encoders, or your own engineered features.
3. **Student architecture / fusion cell**
   - You should edit this cell to define your own model.
4. **Student hyperparameter / training cell**
   - You may tune hidden size, dropout, learning rate, weight decay, batch size, epochs, and patience.

Example fusion idea:

```text
drug_repr → drug_encoder → z_d
protein_repr → protein_encoder → z_p

fusion = [z_d, z_p, z_d * z_p, |z_d - z_p|]
or
fusion = CrossAttention(z_d, z_p)

fusion → predictor → affinity
```

Your model does **not** need to beat the baseline on every split.  
However, you must analyze where it improves and where it fails.

In [ ]:
# Student representation setting: Morgan + MolFormer-final + ESM2-t33 circular sequence-bin selected-layer features.
#
# V6 changes from V5-lite:
#   V5-lite protein_x = 64 bins * 33 ESM layers * 1280 = 2,703,360-d
#   V6 protein_x      = 16 bins * 8 selected ESM layers * 1280 = 163,840-d
#
# Selected ESM2-t33 layers:
#   [4, 8, 12, 16, 20, 24, 28, 33]
#
# Long-protein handling:
#   For sequences longer than ESM_WINDOW_LEN, windows are taken on a conceptual
#   circular protein sequence, so a C-terminal window can wrap around and include
#   the N-terminus. This is a feature-extraction hypothesis for possible
#   N/C-terminal context; it does not use external labels or change splits.
#
# Required companion file:
#   feature_factory.py
# Upload/place it next to this notebook before running this cell.

from pathlib import Path

MODULE_NAME = "feature_factory.py"
assert Path(MODULE_NAME).exists(), (
    f"{MODULE_NAME} not found. Upload it to the same directory as this notebook."
)

from feature_factory import HW3FeatureFactory

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

if device.type != "cuda":
    print("Warning: this ESM2-t33 sequence-bin experiment is intended for a CUDA GPU runtime.")

factory = HW3FeatureFactory(
    dataframe=df,
    dataset_name=DATASET_NAME,
    cache_dir=CACHE_DIR,
    device=device,
)

# V6 speed/idea setting:
#   Drug    = Morgan fingerprint 1024-d + MolFormer final-layer embedding 768-d
#   Protein = ESM2-t33 selected layers, circular sliding-window extraction,
#             residue embeddings pooled into 16 fixed sequence bins.
#
# The old 128-bin all-layer cache cannot be reused for this exact branch because
# both the window definition and the stored layer set changed. A new cache with
# a distinct filename will be created once and then reused.

ESM_SEQUENCE_BINS = 16
ESM_SELECTED_LAYERS = [4, 8, 12, 16, 20, 24, 28, 33]
ESM_WINDOW_LEN = 1020   # conservative residue limit per ESM window
ESM_STRIDE = 511

pack = factory.build_sequence_feature_pack(
    # Drug side
    use_morgan=True,
    morgan_features=drug_features,   # reuse TA Morgan features when available
    use_molformer=True,
    molformer_layers="final",
    molformer_pooling="mean",
    molformer_batch_size=64,
    molformer_device=device,
    molformer_canonicalize=True,
    molformer_remove_isomeric=True,

    # Protein side
    esm_model_size="t33",
    esm_layers=ESM_SELECTED_LAYERS,
    esm_sequence_bins=ESM_SEQUENCE_BINS,
    esm_source_sequence_bins=ESM_SEQUENCE_BINS,
    esm_output_sequence_bins=ESM_SEQUENCE_BINS,
    esm_window_len=ESM_WINDOW_LEN,
    esm_stride=ESM_STRIDE,
    esm_batch_size=1,
    esm_device=device,
    esm_overlap_strategy="position_weighted",
    # None means all selected layers are requested in one ESM forward pass.
    # This avoids repeating the expensive ESM2-t33 forward pass for layer chunks.
    esm_repr_layer_chunk_size=None,
    esm_circular_windows=True,
)

student_drug_features = pack.drug_features
student_protein_features = pack.protein_features

STUDENT_DRUG_DIM = pack.drug_dim
STUDENT_PROTEIN_DIM = pack.protein_dim

# TA-compatible builder is kept for tiny sanity checks only.
# For training this feature, use the id-bank training loop below.
student_xy_builder = pack.make_xy_builder()

print("STUDENT_DRUG_DIM:", STUDENT_DRUG_DIM)
print("STUDENT_PROTEIN_DIM:", STUDENT_PROTEIN_DIM)
print("Feature pack config:", pack.config)

assert STUDENT_DRUG_DIM == 1024 + 768
assert STUDENT_PROTEIN_DIM == ESM_SEQUENCE_BINS * len(ESM_SELECTED_LAYERS) * 1280


In [ ]:
# Reference example only.
# This cell is not used automatically unless you explicitly call ExampleImprovedModel.
# You can read it for ideas, then implement your own model in the next cell.

class ExampleImprovedModel(nn.Module):
    def __init__(
        self,
        drug_dim=STUDENT_DRUG_DIM,
        protein_dim=STUDENT_PROTEIN_DIM,
        proj_dim=256,
        hidden_dim=256,
        dropout=0.2,
    ):
        super().__init__()

        self.drug_dim = drug_dim
        self.protein_dim = protein_dim

        self.drug_encoder = nn.Sequential(
            nn.Linear(drug_dim, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.protein_encoder = nn.Sequential(
            nn.Linear(protein_dim, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        fusion_dim = proj_dim * 4
        self.predictor = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, drug_x, protein_x):
        z_d = self.drug_encoder(drug_x)
        z_p = self.protein_encoder(protein_x)

        fusion = torch.cat([
            z_d,
            z_p,
            z_d * z_p,
            torch.abs(z_d - z_p),
        ], dim=1)

        return self.predictor(fusion).squeeze(-1)

In [ ]:
# Student architecture helper import.
# The actual model class lives in feature_factory.py.

from feature_factory import (
    MyImprovedModel,
    CircularSpliceStyleESMSequenceCNNMorganMolFormerDTA,
    count_trainable_parameters,
)


In [ ]:
# MyImprovedModel summary
# -----------------------
# Imported class: feature_factory.CircularSpliceStyleESMSequenceCNNMorganMolFormerDTA
#
# Expected inputs:
#   drug_x:    [B, 1792]
#              [Morgan 1024-d, MolFormer final-layer 768-d]
#
#   protein_x: [B, 16 * 8 * 1280]
#              flattened [sequence_bin, selected ESM2-t33 layer, ESM hidden]
#
# Inside the model:
#   protein_x -> [B, T=16, L=8, H=1280]
#   circular-padding CNN over ESM layer channels and sequence-bin axis
#   circular-padding dilated residual Conv1d over protein sequence bins
#   drug-conditioned weighted pooling over bins
#   Morgan + sequence-aware protein -> MLP interaction representation
#   interaction representation + MolFormer -> final MLP -> KIBA score

print(MyImprovedModel)


In [ ]:
print("STUDENT_DRUG_DIM:", STUDENT_DRUG_DIM)
print("STUDENT_PROTEIN_DIM:", STUDENT_PROTEIN_DIM)
print("Sequence bins:", pack.config["esm_sequence_bins"])
print("Selected ESM layers:", pack.config["esm_layers"])
print("ESM hidden dim:", pack.config["esm_hidden_dim"])
print("Circular windows:", pack.config["esm_circular_windows"])

assert STUDENT_DRUG_DIM == 1024 + 768
assert STUDENT_PROTEIN_DIM == pack.config["esm_sequence_bins"] * len(pack.config["esm_layers"]) * pack.config["esm_hidden_dim"]

# Small forward-pass sanity check.
test_model = MyImprovedModel(
    drug_dim=STUDENT_DRUG_DIM,
    protein_dim=STUDENT_PROTEIN_DIM,
    morgan_dim=1024,
    sequence_bins=pack.config["esm_sequence_bins"],
    esm_num_layers=len(pack.config["esm_layers"]),
    esm_hidden_dim=pack.config["esm_hidden_dim"],
    layer_cnn_channels=96,
    layer_cnn_dilations=(1, 2, 4, 8),
    seq_model_dim=384,
    seq_dilations=(1, 2, 4, 8),
    proj_dim=384,
    hidden_dim=768,
    dropout=0.20,
    protein_layer_dropout=0.05,
    residual_final_layer=True,
    circular_padding=True,
).to(device)

print("Trainable parameters:", count_trainable_parameters(test_model))

drug_batch = torch.randn(1, STUDENT_DRUG_DIM, device=device)
protein_batch = torch.randn(1, STUDENT_PROTEIN_DIM, device=device)

with torch.no_grad():
    y = test_model(drug_batch, protein_batch)

print("Output shape:", y.shape)
assert y.shape == (1,)

del test_model, drug_batch, protein_batch, y
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# Student training: memory-efficient id-bank loop for ESM2-t33 circular sequence-bin features.
#
# Important:
# - This does not change split construction.
# - Label standardization still uses train statistics only.
# - Metrics are computed by the notebook's original compute_metrics function.
# - The difference from TA train_one_split is only that feature banks are indexed by ID
#   instead of expanding protein_x to pair-level matrices in RAM.

from feature_factory import make_feature_banks, train_one_split_idbank
from pathlib import Path

# Save a complete PyTorch model object after each split finishes.
# Paths will be:
#   kiba_saved_models_splice_v6_circular16_sel8_savefull/random/model.pth
#   kiba_saved_models_splice_v6_circular16_sel8_savefull/cold_drug/model.pth
#   kiba_saved_models_splice_v6_circular16_sel8_savefull/cold_target/model.pth
MODEL_SAVE_ROOT = Path(f"{DATASET_NAME.lower()}_saved_models_splice_v6_circular16_sel8_savefull")
MODEL_SAVE_ROOT.mkdir(parents=True, exist_ok=True)

MY_MODEL_CONFIG = {
    "epochs": 60,
    "batch_size": 64,   # V6 protein_x is ~16.5x smaller than V5-lite; use 32 if OOM.
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "patience": 10,
    "max_grad_norm": 1.0,
    "amp": True,
    # Keep feature banks on GPU in AMP dtype, while loss stays float32.
    "bank_device_dtype": "amp",
    "save_model_dir": MODEL_SAVE_ROOT,
    "save_model_filename": "model.pth",
    "scheduler_config": {
        "type": "onecycle",
        "pct_start": 0.10,
        "div_factor": 10.0,
        "final_div_factor": 100.0,
    },
    "model_kwargs": {
        "morgan_dim": 1024,
        "sequence_bins": pack.config["esm_sequence_bins"],
        "esm_num_layers": len(pack.config["esm_layers"]),
        "esm_hidden_dim": pack.config["esm_hidden_dim"],
        "layer_cnn_channels": 96,
        "layer_cnn_dilations": (1, 2, 4, 8),
        "seq_model_dim": 384,
        "seq_dilations": (1, 2, 4, 8),
        "proj_dim": 384,
        "hidden_dim": 768,
        "dropout": 0.20,
        "protein_layer_dropout": 0.05,
        "residual_final_layer": True,
        "circular_padding": True,
    },
}

# Build compact banks once. This avoids materializing pair-level protein matrices.
drug_bank, protein_bank, drug_to_id, protein_to_id = make_feature_banks(
    student_drug_features,
    student_protein_features,
)

RUN_MY_MODEL = True

my_results = []
my_histories = {}

if RUN_MY_MODEL:
    for split_name, split in splits.items():
        result, model, history_df = train_one_split_idbank(
            split_name=split_name,
            split=split,
            model_class=MyImprovedModel,
            drug_bank=drug_bank,
            protein_bank=protein_bank,
            drug_to_id=drug_to_id,
            protein_to_id=protein_to_id,
            compute_metrics_fn=compute_metrics,
            device=device,
            **MY_MODEL_CONFIG,
        )

        result["Model"] = (
            f"Morgan_MolFormerFinal_ESM2t33CircularSeqCNN_"
            f"bins{pack.config['esm_sequence_bins']}_layers{len(pack.config['esm_layers'])}"
        )
        my_results.append(result)
        my_histories[split_name] = history_df

        if device.type == "cuda":
            torch.cuda.empty_cache()

    my_results_df = pd.DataFrame(my_results)
    my_results_df = my_results_df[
        [
            "Model",
            "Split",
            "RMSE",
            "MAE",
            "Pearson",
            "Spearman",
            "Valid_RMSE",
            "Valid_MAE",
            "Valid_Pearson",
            "Valid_Spearman",
            "BestValidLoss_standardized_MSE",
            "EpochsTrained",
            "SavedModelPath",
        ]
    ]

    display(my_results_df)

    my_results_path = f"{DATASET_NAME.lower()}_my_model_results_splice_v6_circular16_sel8.csv"
    my_results_df.to_csv(my_results_path, index=False)
    print(f"Saved {my_results_path}")
else:
    print("RUN_MY_MODEL is False.")


# 10. Compare baseline and your model

After running your model, create a combined result table.

Your table should look like:

| Model | Split | RMSE | MAE | Pearson | Spearman |
|---|---|---:|---:|---:|---:|
| BaselineMLP | random | | | | |
| BaselineMLP | cold_drug | | | | |
| BaselineMLP | cold_target | | | | |
| MyImprovedModel | random | | | | |
| MyImprovedModel | cold_drug | | | | |
| MyImprovedModel | cold_target | | | | |

In [ ]:
# Run this after you have trained MyImprovedModel.

if "my_results_df" in globals():
    combined_results_df = pd.concat([baseline_results_df, my_results_df], ignore_index=True)
    display(combined_results_df)
    combined_results_path = f"{DATASET_NAME.lower()}_combined_results.csv"
    combined_results_df.to_csv(combined_results_path, index=False)
    print(f"Saved {combined_results_path}")
else:
    print("my_results_df does not exist yet. Train MyImprovedModel first.")

# 12. Report questions

Answer the following questions directly in this notebook.

---

## Q1. Model improvement

Describe the improvement you applied to the baseline model.

Your answer should include:

- What did you change?
- Why did you choose this improvement?

Examples:

- architecture change
- fusion method
- hidden dimension / dropout / weight decay
- optimizer or learning rate schedule
- regularization strategy
- feature processing

**Answer:**

---

## Q2. Split-wise performance comparison

Compare the baseline and your improved model under each split.

Discuss the results for:

- Random split
- Cold drug split
- Cold target split

For each split, explain whether your model improved, stayed similar, or became worse.
Do not only report the table. Explain the pattern in the metrics.

**Answer:**

---

## Q3. Effect of your improvement

What effect do you think your improvement had on the model?

Your discussion should connect your design choice to the observed results.

Examples:

- Did it reduce overfitting?
- Did it improve drug–protein feature interaction?
- Did it help only in random split but not in cold-start splits?
- Did it make the model more stable?
- Did it increase model capacity too much?

**Answer:**

# 13. Grading rubric

Base score: **90 points**

| Component | Points |
|---|---:|
| Baseline execution and result reporting | 25 |
| Model improvement implementation | 30 |
| Split-wise performance comparison | 20 |
| Discussion quality | 15 |
| Base total | 90 |

- Hyperparameter-only modifications can receive partial credit, but they are not sufficient for full credit in the model improvement section.
- To receive a higher score in the model improvement section, your work should focus on meaningful architectural changes rather than only hyperparameter tuning.

Bonus: **10 points**

| Bonus item | Points |
|---|---:|
| Uses a meaningful method for cold-setting generalization | +4 |
| Implements a meaningful interaction-aware fusion strategy | +3 |
| Provides better handling of long protein sequences | +3 |
| Bonus total | +10 |